# 17 — The Retention ROI Copilot

Every capable version of this project answers *"who is likely to leave, and
why?"*. That is where the Build Notes stop, and it is where the field stops.

**It is not the question an HR director actually has.** Theirs is: *"I have a
retention budget. Who do I spend it on, on what, and what do I get back?"*

This notebook covers the three components that answer it.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


---
## A · Cross-population transfer — measured, not assumed

Finding F1 said the two datasets are different companies. That is a constraint —
and also an opportunity almost no project at this scale gets: **Population B
carries its own attrition label**, so the model can be *externally validated*
instead of merely hoped over.

In [2]:
from hrai.ml.transfer import run_transfer_validation
import json

result = run_transfer_validation()
a = result["population_a"]["held_out_metrics"]; b = result["population_b"]["external_metrics"]
pd.DataFrame({"Population A (held-out)": a, "Population B (external)": b}).loc[
    ["roc_auc", "pr_auc", "precision", "recall", "f1", "brier"]]

2026-08-28 01:56:12 | INFO  | pipeline built


2026-08-28 01:56:12 | INFO  | cross-population transfer validated


,Population A (held-out),Population B (external)
roc_auc,0.6250,0.5039
pr_auc,0.2747,0.1038
precision,0.2917,0.1028
recall,0.2979,0.1028
f1,0.2947,0.1028
brier,0.1308,0.1062


### The honest result: the model does not transfer

ROC-AUC **0.50** on Population B — indistinguishable from chance. This is a
negative result, and reporting it is worth more than a fudged positive: it is
independent confirmation of finding F1 by a completely different method.

In [3]:
pd.DataFrame(result["distribution_shift"])

,feature,psi,shift,population_a_mean,population_b_mean
0,YearsAtCompany,5.4199,severe,7.01,2.91
1,Age,0.9673,severe,36.92,47.68
2,WorkLifeBalance,0.8790,severe,2.76,2.49
3,JobSatisfaction,0.2131,moderate,2.73,2.52
4,Gender,0.1043,moderate,NaN,NaN


In [4]:
print(result["interpretation"])

The model does not transfer: on Population B it performs at close to chance. Attrition risk for Population B should be reported as unavailable rather than estimated. Severe distribution shift on YearsAtCompany, Age, WorkLifeBalance explains much of the gap and is the first thing to address.


Tenure PSI of **5.4** is the diagnosis: Population A averages 7 years of
tenure, Population B 2.9. These workforces are not comparable, and the system
therefore withholds a risk score for Population B rather than inventing one.

---
## B · Counterfactual interventions

For each at-risk employee, perturb one *actionable* lever through the fitted
pipeline and measure the change in predicted risk. Protected attributes — age,
gender, marital status — are excluded structurally, and a test asserts it.

In [5]:
from hrai.intelligence.counterfactual import CounterfactualEngine, load_levers

for lever in load_levers():
    print(f"  {lever.label:32} {lever.feature:24} {lever.rationale}")
print("\nprotected attributes, never levers:", get("retention_roi.protected_attributes"))

  Remove mandatory overtime        OverTime                 Backfilling the hours the employee no longer works.
  Work-life balance programme      WorkLifeBalance          Flexible working, workload rebalancing, manager coaching.
  Targeted upskilling              TrainingTimesLastYear    Priced from observed Training Cost in the engagement data.
  Promotion                        YearsSinceLastPromotion  Salary band uplift carried for a year.
  Compensation increase            MonthlyIncome            A 10% raise costs 1.2 months of salary over a year.
  Additional stock options         StockOptionLevel         One additional stock option band.

protected attributes, never levers: ['Age', 'Gender', 'MaritalStatus', 'Over18']


In [6]:
engine = CounterfactualEngine()
df = load_processed("employee_attrition_processed").drop(columns=["attrition_flag"])
risk = engine.model.predict_proba(df)[:, 1]
row = df.iloc[[int(np.argmax(risk))]]

plan = engine.plan_for(row)
print(f"Employee A-{plan.employee_id}  baseline risk {plan.baseline_risk:.1%} "
      f"({plan.risk_band})  replacement cost {plan.replacement_cost:,.0f}\n")
pd.DataFrame([i.to_dict() for i in plan.single_lever])[
    ["label", "from_value", "to_value", "new_risk", "risk_reduction",
     "cost", "expected_value_saved", "roi"]]

2026-08-28 01:56:12 | INFO  | model loaded


2026-08-28 01:56:12 | INFO  | course catalogue built


Employee A-622  baseline risk 93.5% (HIGH)  replacement cost 14,040


,label,from_value,to_value,new_risk,risk_reduction,cost,expected_value_saved,roi
0,Remove mandatory overtime,Yes,No,0.8131,0.1224,2808.0,1718.0693,0.6118
1,Work-life balance programme,1,2,0.9200,0.0155,1170.0,217.6589,0.1860
2,Targeted upskilling,3,4,0.9289,0.0066,576.1,91.9640,0.1596
3,Additional stock options,0,1,0.9241,0.0113,1404.0,159.1967,0.1134
4,Compensation increase,2340,2574,0.9343,0.0012,2808.0,16.9488,0.0060


---
## C · The budget-constrained action plan

A 0/1 knapsack across the workforce, solved greedily by ROI: *"with this budget,
these are the interventions, on these people, that maximise expected value
retained."*

In [7]:
from hrai.intelligence.counterfactual import build_action_plan

plan = build_action_plan(budget=500_000, engine=engine, min_risk=0.30, max_employees=250)
pd.Series({
    "budget": f"{plan['budget']:,.0f}",
    "spend": f"{plan['spend']:,.0f}",
    "employees at risk": plan["employees_at_risk"],
    "employees funded": plan["employees_covered"],
    "unfunded": plan["unfunded_at_risk"],
    "expected exits prevented": plan["expected_attritions_prevented"],
    "expected value retained": f"{plan['expected_value_retained']:,.0f}",
    "return on investment": f"{plan['return_on_investment']}x",
})

2026-08-28 01:56:13 | INFO  | action plan candidates selected


2026-08-28 01:57:02 | INFO  | action plan built


budget                      500,000
spend                       499,777
employees at risk               242
employees funded                235
unfunded                          7
expected exits prevented      32.53
expected value retained     677,275
return on investment          1.36x
dtype: object

In [8]:
pd.DataFrame(plan["interventions"]).head(12)[
    ["person_key", "baseline_risk", "label", "new_risk", "cost", "roi"]]

,person_key,baseline_risk,label,new_risk,cost,roi
0,A-1277,0.3962,Targeted upskilling,0.3718,576.1,4.7811
1,A-1740,0.3714,Targeted upskilling,0.3485,576.1,4.7286
2,A-1307,0.4458,Targeted upskilling,0.4225,576.1,4.1519
3,A-363,0.3385,Targeted upskilling,0.3154,576.1,4.0591
4,A-1372,0.5035,Targeted upskilling,0.4773,576.1,3.7389
5,A-1975,0.4955,Targeted upskilling,0.4696,576.1,3.6013
6,A-1716,0.5240,Targeted upskilling,0.4987,576.1,3.4051
7,A-2055,0.4950,Targeted upskilling,0.4685,576.1,2.9994
8,A-970,0.5617,Targeted upskilling,0.5368,576.1,2.7433
9,A-1167,0.7585,Targeted upskilling,0.7395,576.1,2.7293


---
## D · Fairness audit

A model that influences decisions about people needs a bias check.

In [9]:
from hrai.ml.fairness import audit

report = audit()
for attribute, res in report["attributes"].items():
    print(f"{attribute:16} equal-opportunity diff {res['equal_opportunity_difference']:<8} "
          f"{'PASS' if res['within_tolerance'] else 'FLAG'}")
    print(f"   {report['interpretation'][attribute][:150]}...\n")

2026-08-28 01:57:02 | INFO  | model loaded


2026-08-28 01:57:02 | INFO  | fairness audit complete


Gender           equal-opportunity diff 0.032    PASS
   Within tolerance on both equal opportunity and predictive equality....

MaritalStatus    equal-opportunity diff 0.1007   FLAG
   Flagged, but the cause is a genuine base-rate difference of 15.4% across groups, not miscalibration — every group's calibration gap is under 0.022. Th...

AgeBand          equal-opportunity diff 0.2064   FLAG
   Flagged, but the cause is a genuine base-rate difference of 18.2% across groups, not miscalibration — every group's calibration gap is under 0.040. Th...



### Guardrails, shipped in the product not the README

* Association-based decision support, **not causal inference**. The model learned
  which employees historically left, not what would have changed had a lever been
  pulled.
* Counterfactuals assume features move independently — true enough for single
  levers, weaker for combinations, which are flagged as such.
* Every API response and dashboard panel carries the caveat, and a human approves
  before anything is acted on.

An HR tool that overstates its certainty is a liability. Saying so is part of the
deliverable.